# Notebook 1: SMILES Strings & RDKit Fundamentals

**Goal:** Understand what SMILES strings are, how to load and manipulate molecules using RDKit, and how to extract basic chemical properties — all from a bioinformatics perspective.

**Dataset:** ESOL (Delaney) — water solubility measurements for ~1,100 small molecules. This is one of the most commonly used benchmark datasets in cheminformatics. Each molecule has:
- A **SMILES string** (the molecule's text representation)
- A **log solubility** value (a continuous property we can later predict)

---

## Background: What is a SMILES string?

In bioinformatics, you're used to sequences — DNA, RNA, and proteins are all linear strings of characters representing biological molecules. SMILES (Simplified Molecular Input Line Entry System) takes the same idea and applies it to *chemical* molecules.

A SMILES string encodes a molecule's atoms and bonds as a plain-text string:

| Molecule     | SMILES           | What it means                        |
|--------------|------------------|--------------------------------------|
| Methane      | `C`              | One carbon atom                      |
| Ethanol      | `CCO`            | Two carbons, one oxygen              |
| Benzene      | `c1ccccc1`       | 6 aromatic carbons in a ring         |
| Aspirin      | `CC(=O)Oc1ccccc1C(=O)O` | Full aspirin structure     |

**Key SMILES syntax rules:**
- Uppercase letters = atoms (C, N, O, S, etc.)
- Lowercase letters = aromatic atoms (c, n, o, etc.)
- `=` = double bond, `#` = triple bond (single bond is implicit)
- Parentheses `()` = branches off the main chain
- Numbers = ring closures (atom 1 connects back to where you first saw that number)
- `[...]` = atoms that need special notation (charge, isotope, hydrogen count)

Just like how you'd use Biopython to parse a FASTA file into sequence objects, you'll use **RDKit** to parse a SMILES string into a molecule object you can analyze.

---

## Step 0: Install and import libraries

RDKit is the gold-standard open-source cheminformatics library. Think of it as the BioPython/Scanpy of chemistry.

In [ ]:
# If running for the first time, install rdkit:
# !pip install rdkit

# Core data handling
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# RDKit — the chemistry library
from rdkit import Chem                        # Core molecule handling
from rdkit.Chem import Descriptors            # Molecular descriptor calculation
from rdkit.Chem import Draw                   # For drawing/visualizing molecules
from rdkit.Chem import rdMolDescriptors       # More descriptor options
from rdkit.Chem.Draw import IPythonConsole    # Renders molecule images inline in Jupyter

print("All libraries imported successfully!")

## Step 1: Load the ESOL Dataset

The ESOL dataset contains water solubility measurements for small organic molecules. Each row is one molecule. The key columns are:
- `smiles` — the SMILES string representing the molecule
- `measured log solubility in mols per litre` — the target variable (what we'd eventually predict)

We load it directly from GitHub using pandas — just like loading any CSV.

In [ ]:
# Load the ESOL dataset directly from GitHub
# This is a CSV where each row is one molecule
ESOL_URL = "https://raw.githubusercontent.com/GLambard/Molecules_Dataset_Collection/master/latest/ESOL_delaney-processed.csv"

df = pd.read_csv(ESOL_URL)

# Let's see what we're working with
print(f"Dataset shape: {df.shape}")
print(f"\nColumn names:")
for col in df.columns:
    print(f"  - {col}")

In [ ]:
# Preview the first few rows
# Notice that 'smiles' is just a regular string column — plain text!
df.head()

In [ ]:
# Let's rename the target column to something shorter and easier to type
# The original name is very long
df = df.rename(columns={"measured log solubility in mols per litre": "log_solubility"})

# Print a few SMILES strings so we can see what they look like as raw text
print("Sample SMILES strings from the dataset:")
print()
for i, row in df.head(5).iterrows():
    print(f"  Compound: {row['Compound ID']}")
    print(f"  SMILES:   {row['smiles']}")
    print(f"  Log Sol:  {row['log_solubility']:.3f}")
    print()

## Step 2: Parsing SMILES → Molecule Objects

Right now `smiles` is just a string column in pandas — like any other text. To do chemistry with it, we need to **parse** it into an RDKit molecule object. This is analogous to parsing a FASTA string into a BioPython `SeqRecord`.

The key function is: `Chem.MolFromSmiles(smiles_string)`

It returns:
- A **molecule object** if the SMILES is valid
- `None` if the SMILES is invalid or unparseable (important to handle this!)

In [ ]:
# Parse a single SMILES string to see what we get back
ethanol_smiles = "CCO"
ethanol_mol = Chem.MolFromSmiles(ethanol_smiles)

print(f"Input SMILES: {ethanol_smiles}")
print(f"Type of result: {type(ethanol_mol)}")
print(f"Number of atoms: {ethanol_mol.GetNumAtoms()}")
print(f"Number of bonds: {ethanol_mol.GetNumBonds()}")

In [ ]:
# You can also draw the molecule directly in the notebook
# This renders an image of the chemical structure
from rdkit.Chem import Draw

ethanol_mol  # In Jupyter, just evaluating an RDKit mol object shows its structure

In [ ]:
# Let's draw a few molecules side-by-side to build intuition for what SMILES look like
example_smiles = [
    ("Methane",   "C"),
    ("Ethanol",   "CCO"),
    ("Benzene",   "c1ccccc1"),
    ("Aspirin",   "CC(=O)Oc1ccccc1C(=O)O"),
    ("Caffeine",  "Cn1cnc2c1c(=O)n(c(=O)n2C)C"),
    ("Ibuprofen", "CC(C)Cc1ccc(cc1)C(C)C(=O)O"),
]

# Parse each SMILES into a molecule object
mols = []
labels = []
for name, smi in example_smiles:
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:          # Always check that parsing succeeded!
        mols.append(mol)
        labels.append(name)

# Draw all of them in a grid
img = Draw.MolsToGridImage(
    mols,
    molsPerRow=3,
    subImgSize=(300, 200),
    legends=labels
)
img

## Step 3: Apply SMILES Parsing to the Full Dataset

Now let's parse every SMILES in the ESOL dataset and add the molecule objects as a new column. This is a standard operation you'll do at the start of almost any cheminformatics workflow — it's the equivalent of loading sequences into memory before processing them.

In [ ]:
# Apply Chem.MolFromSmiles to every row in the 'smiles' column
# .apply() here works just like it does with any pandas string column
df["mol"] = df["smiles"].apply(Chem.MolFromSmiles)

# Count how many molecules parsed successfully vs failed
n_total   = len(df)
n_valid   = df["mol"].notna().sum()    # mol is not None
n_invalid = df["mol"].isna().sum()     # mol is None (parsing failed)

print(f"Total molecules:          {n_total}")
print(f"Successfully parsed:      {n_valid}")
print(f"Failed to parse (None):   {n_invalid}")

In [ ]:
# Always drop rows where parsing failed before doing any downstream work
# A None molecule object will crash most RDKit functions
df = df[df["mol"].notna()].reset_index(drop=True)

print(f"Remaining molecules after filtering: {len(df)}")
print()
print("Updated DataFrame columns:")
print(df.dtypes)

## Step 4: Computing Basic Molecular Descriptors

**Molecular descriptors** are numerical properties calculated from a molecule's structure. In a machine learning context, these are your *features* — the numbers you'd feed into a model to predict something like solubility or toxicity.

This is analogous to computing gene expression values or chromatin accessibility scores from a biological sequence — you're converting a complex object into a set of interpretable numbers.

Common descriptors you'll encounter in drug discovery:

| Descriptor | Abbreviation | What it means |
|---|---|---|
| Molecular Weight | MolWt | Total mass of the molecule |
| LogP | MolLogP | Oil/water partition coefficient — how "greasy" a molecule is |
| H-Bond Donors | NumHDonors | Number of N-H or O-H groups (can donate H-bonds) |
| H-Bond Acceptors | NumHAcceptors | Number of N or O atoms (can accept H-bonds) |
| Topological Polar Surface Area | TPSA | Related to membrane permeability |
| Rotatable Bonds | NumRotatableBonds | Molecular flexibility |
| Ring Count | RingCount | Number of ring structures |

These 5-7 descriptors are the basis of **Lipinski's Rule of Five** — the classic druglikeness filter used in pharma.

In [ ]:
# Compute individual descriptors from a single molecule first, to understand the API
aspirin = Chem.MolFromSmiles("CC(=O)Oc1ccccc1C(=O)O")

print("Aspirin molecular descriptors:")
print(f"  Molecular Weight:       {Descriptors.MolWt(aspirin):.2f} Da")
print(f"  LogP (lipophilicity):   {Descriptors.MolLogP(aspirin):.2f}")
print(f"  H-Bond Donors:          {rdMolDescriptors.CalcNumHBD(aspirin)}")
print(f"  H-Bond Acceptors:       {rdMolDescriptors.CalcNumHBA(aspirin)}")
print(f"  TPSA:                   {Descriptors.TPSA(aspirin):.2f} Å²")
print(f"  Rotatable Bonds:        {rdMolDescriptors.CalcNumRotatableBonds(aspirin)}")
print(f"  Ring Count:             {rdMolDescriptors.CalcNumRings(aspirin)}")

In [ ]:
# Now compute descriptors for every molecule in the dataset
# We define a helper function that takes a mol object and returns a dict of descriptors
# Using a function like this keeps the code clean and easy to extend

def compute_descriptors(mol):
    """
    Compute a standard set of molecular descriptors from an RDKit molecule object.
    
    Parameters
    ----------
    mol : rdkit.Chem.Mol
        A parsed RDKit molecule object (must not be None)
    
    Returns
    -------
    dict
        A dictionary of descriptor name → value
    """
    return {
        "mol_weight":       Descriptors.MolWt(mol),
        "logP":             Descriptors.MolLogP(mol),
        "hbd":              rdMolDescriptors.CalcNumHBD(mol),       # H-bond donors
        "hba":              rdMolDescriptors.CalcNumHBA(mol),       # H-bond acceptors
        "tpsa":             Descriptors.TPSA(mol),
        "rotatable_bonds":  rdMolDescriptors.CalcNumRotatableBonds(mol),
        "ring_count":       rdMolDescriptors.CalcNumRings(mol),
        "num_atoms":        mol.GetNumAtoms(),
    }

# Apply the function to every row — this returns a Series of dicts
# pd.DataFrame() on that Series expands each dict into columns
descriptor_df = pd.DataFrame(
    df["mol"].apply(compute_descriptors).tolist()
)

print(f"Descriptor matrix shape: {descriptor_df.shape}")
descriptor_df.head()

In [ ]:
# Merge the descriptors back into the main dataframe
# This is a standard pandas operation — just concatenate along columns (axis=1)
df = pd.concat([df, descriptor_df], axis=1)

print(f"Final dataframe shape: {df.shape}")
df.head()

## Step 5: Exploring the Data — Distributions and Relationships

Before any ML work, it's essential to understand your data. Let's look at the distributions of descriptors and how they relate to the target variable (log solubility).

This is just standard EDA — nothing chemistry-specific here. Your bioinformatics instincts apply directly.

In [ ]:
# Summary statistics for descriptors and target
cols_to_describe = ["log_solubility", "mol_weight", "logP", "hbd", "hba", "tpsa", "rotatable_bonds"]
df[cols_to_describe].describe().round(2)

In [ ]:
# Plot the distribution of log solubility (our target variable)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram of log solubility
axes[0].hist(df["log_solubility"], bins=40, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].set_xlabel("Log Solubility (mol/L)")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of Log Solubility (Target Variable)")
axes[0].axvline(df["log_solubility"].mean(), color="red", linestyle="--", label=f"Mean: {df['log_solubility'].mean():.2f}")
axes[0].legend()

# Histogram of LogP (lipophilicity)
axes[1].hist(df["logP"], bins=40, color="darkorange", edgecolor="white", alpha=0.8)
axes[1].set_xlabel("LogP (lipophilicity)")
axes[1].set_ylabel("Count")
axes[1].set_title("Distribution of LogP")
axes[1].axvline(df["logP"].mean(), color="red", linestyle="--", label=f"Mean: {df['logP'].mean():.2f}")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: LogP vs log solubility
# This is one of the most important relationships in cheminformatics:
# more lipophilic (higher LogP) molecules tend to be LESS water-soluble

plt.figure(figsize=(8, 5))
plt.scatter(
    df["logP"],
    df["log_solubility"],
    alpha=0.4,
    s=20,
    color="steelblue"
)
plt.xlabel("LogP (lipophilicity)")
plt.ylabel("Log Solubility (mol/L)")
plt.title("LogP vs Solubility\n(More lipophilic = less soluble, as expected)")

# Compute and display the correlation
correlation = df["logP"].corr(df["log_solubility"])
plt.text(0.05, 0.05, f"Pearson r = {correlation:.2f}", transform=plt.gca().transAxes,
         fontsize=12, color="darkred")
plt.tight_layout()
plt.show()

print(f"Pearson correlation between LogP and log solubility: {correlation:.3f}")
print("Negative correlation makes chemical sense: greasier molecules dissolve less in water.")

In [ ]:
# Correlation heatmap across all descriptors and target
import matplotlib.pyplot as plt

corr_cols = ["log_solubility", "mol_weight", "logP", "hbd", "hba", "tpsa", "rotatable_bonds", "ring_count"]
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, label="Pearson Correlation")

# Label the axes
ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha="right")
ax.set_yticklabels(corr_cols)

# Annotate each cell with the correlation value
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        ax.text(j, i, f"{corr_matrix.iloc[i, j]:.2f}",
                ha="center", va="center", fontsize=8,
                color="black" if abs(corr_matrix.iloc[i, j]) < 0.7 else "white")

ax.set_title("Correlation Matrix: Molecular Descriptors vs Log Solubility")
plt.tight_layout()
plt.show()

## Step 6: Visualizing Molecules from the Dataset

Let's look at the actual molecular structures of the most and least soluble compounds in the dataset. This builds intuition for what solubility looks like structurally.

In [ ]:
# Get the 6 most soluble compounds
most_soluble = df.nlargest(6, "log_solubility")

mols   = most_soluble["mol"].tolist()
labels = [
    f"{row['Compound ID']}\nlogS={row['log_solubility']:.2f}"
    for _, row in most_soluble.iterrows()
]

img = Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(300, 200), legends=labels)
print("6 Most Soluble Compounds:")
img

In [ ]:
# Get the 6 least soluble compounds
least_soluble = df.nsmallest(6, "log_solubility")

mols   = least_soluble["mol"].tolist()
labels = [
    f"{row['Compound ID']}\nlogS={row['log_solubility']:.2f}"
    for _, row in least_soluble.iterrows()
]

img = Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(300, 200), legends=labels)
print("6 Least Soluble Compounds:")
img

## Step 7: Canonical SMILES and Standardization

One subtlety that often trips people up: **the same molecule can be written as many different SMILES strings**.

For example, ethanol can be written as:
- `CCO`
- `OCC`
- `C(O)C`

All three are the same molecule! This is a major source of duplicates in chemical databases.

RDKit solves this with **canonical SMILES** — a standardized, unique representation of any molecule. This is the chemistry equivalent of normalizing gene names or using canonical transcript IDs.

In [ ]:
# Demonstrate that different SMILES can represent the same molecule
ethanol_variants = ["CCO", "OCC", "C(O)C", "[CH3][CH2][OH]"]

print("Ethanol written different ways — all produce the same canonical SMILES:")
print()
for smi in ethanol_variants:
    mol = Chem.MolFromSmiles(smi)
    canonical = Chem.MolToSmiles(mol)   # Convert mol object back to canonical SMILES
    print(f"  Input:     {smi:<20}  →  Canonical: {canonical}")

In [ ]:
# Always canonicalize SMILES in your data pipeline to avoid duplicate molecules
# This is a best practice in cheminformatics

def canonicalize_smiles(smi):
    """Convert a SMILES string to its canonical (standardized) form.
    Returns None if the SMILES is invalid."""
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol)

df["canonical_smiles"] = df["smiles"].apply(canonicalize_smiles)

# Check for duplicates after canonicalization
n_duplicates = df["canonical_smiles"].duplicated().sum()
print(f"Duplicate molecules found after canonicalization: {n_duplicates}")
print()
print("Original SMILES vs Canonical SMILES (first 5 rows):")
df[["smiles", "canonical_smiles"]].head()

## Step 8: Lipinski's Rule of Five

Lipinski's Rule of Five is a famous druglikeness filter used in pharmaceutical research to assess whether a compound is likely to be orally bioavailable. Understanding it is important context for working at a company like Valo.

A molecule is considered **drug-like** if it satisfies **most** of these criteria:
1. Molecular Weight ≤ 500 Da
2. LogP ≤ 5
3. H-Bond Donors ≤ 5
4. H-Bond Acceptors ≤ 10

Let's apply this filter to the ESOL dataset.

In [ ]:
def lipinski_pass(row):
    """
    Apply Lipinski's Rule of Five filter.
    Returns True if the molecule passes (drug-like), False otherwise.
    """
    mw_ok  = row["mol_weight"] <= 500
    logp_ok = row["logP"] <= 5
    hbd_ok  = row["hbd"] <= 5
    hba_ok  = row["hba"] <= 10
    
    # Must pass at least 3 of 4 criteria ("rule of five" allows one violation)
    return sum([mw_ok, logp_ok, hbd_ok, hba_ok]) >= 3

df["lipinski_pass"] = df.apply(lipinski_pass, axis=1)

n_pass = df["lipinski_pass"].sum()
n_fail = (~df["lipinski_pass"]).sum()
print(f"Molecules passing Lipinski's Rule of Five: {n_pass} ({100*n_pass/len(df):.1f}%)")
print(f"Molecules failing:                         {n_fail} ({100*n_fail/len(df):.1f}%)")

In [ ]:
# Do Lipinski-passing molecules differ in solubility?
pass_group = df[df["lipinski_pass"]]["log_solubility"]
fail_group = df[~df["lipinski_pass"]]["log_solubility"]

print("Log Solubility by Lipinski Status:")
print(f"  Passing molecules — mean: {pass_group.mean():.2f}, median: {pass_group.median():.2f}")
print(f"  Failing molecules — mean: {fail_group.mean():.2f}, median: {fail_group.median():.2f}")

# Box plot comparison
plt.figure(figsize=(6, 4))
plt.boxplot(
    [pass_group, fail_group],
    labels=["Lipinski Pass", "Lipinski Fail"],
    patch_artist=True,
    boxprops=dict(facecolor="steelblue", alpha=0.6)
)
plt.ylabel("Log Solubility (mol/L)")
plt.title("Solubility by Lipinski Filter Status")
plt.tight_layout()
plt.show()

## Summary

In this notebook, you learned:

1. **What SMILES strings are** — text representations of molecular structures, analogous to biological sequences
2. **How to parse SMILES with RDKit** — `Chem.MolFromSmiles()` converts text → molecule object
3. **How to handle parsing failures** — always check for `None` before computing properties
4. **How to compute molecular descriptors** — numerical features extracted from structure (MW, LogP, HBD, HBA, TPSA)
5. **How to canonicalize SMILES** — standardize representations to detect duplicates
6. **Lipinski's Rule of Five** — the classic druglikeness filter

### What's next?

→ **Notebook 2** covers the SDF file format — storing multiple molecules with associated data in a single file, which is the most common format you'll encounter in drug discovery workflows.